# AInki — Fine-tune Mistral 7B with QLoRA

Fine-tune Mistral-7B-Instruct on golden-standard Q&A pairs for spaced repetition.

**Setup:**
- Attach `training_dataset_filtered.jsonl` as a Kaggle dataset input
- Enable GPU: Settings → Accelerator → GPU T4 x2
- Add Kaggle secrets: `WANDB_API_KEY`, `HF_TOKEN`

In [ ]:
!pip install -q transformers accelerate peft trl bitsandbytes datasets wandb

In [ ]:
import json
import os
from pathlib import Path

import torch
import wandb
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## Config

In [ ]:
# ── Paths ──
# Update this to match your Kaggle input dataset name
DATASET_PATH = "/kaggle/input/ainki-training-data/training_dataset_filtered.jsonl"
OUTPUT_DIR = "/kaggle/working/finetuned_model"

# ── Model ──
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"  # fits on T4 16GB with QLoRA

# ── Training ──
EPOCHS = 3
BATCH_SIZE = 1
GRAD_ACCUM = 8
LEARNING_RATE = 2e-4
MAX_SEQ_LEN = 4096

# ── LoRA ──
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

## Login to W&B and HuggingFace

In [ ]:
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()

# W&B
wandb_key = secrets.get_secret("WANDB_API_KEY")
if wandb_key:
    wandb.login(key=wandb_key)
    REPORT_TO = "wandb"
    print("W&B logged in")
else:
    REPORT_TO = "none"
    print("No WANDB_API_KEY found, skipping W&B")

# HuggingFace (needed to download gated Mistral models)
hf_token = secrets.get_secret("HF_TOKEN")
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    print("HF token set")
else:
    print("No HF_TOKEN found — if model is gated, download will fail")

## Load dataset

In [ ]:
def load_jsonl(path: str) -> Dataset:
    examples = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                examples.append(json.loads(line))
    return Dataset.from_list(examples)

dataset = load_jsonl(DATASET_PATH)
print(f"Training examples: {len(dataset)}")
print(f"Columns: {dataset.column_names}")
print(f"\nFirst example messages[0]: {dataset[0]['messages'][0]}")

## Load tokenizer & model

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

# Apply chat template
def format_chat(example: dict) -> dict:
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

dataset = dataset.map(format_chat, remove_columns=dataset.column_names)
print(f"Sample (first 500 chars):\n{dataset[0]['text'][:500]}")

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    attn_implementation="eager",
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

## LoRA config

In [ ]:
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Train

In [ ]:
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    bf16=use_bf16,
    fp16=not use_bf16 and torch.cuda.is_available(),
    logging_steps=1,
    save_strategy="epoch",
    max_seq_length=MAX_SEQ_LEN,
    dataset_text_field="text",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",
    report_to=REPORT_TO,
    run_name="ainki-mistral-7b-finetune",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)

print("Starting training...")
trainer.train()

## Save adapter weights

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Adapter weights saved to {OUTPUT_DIR}")
print(f"Files: {os.listdir(OUTPUT_DIR)}")

## Quick test

In [ ]:
from peft import AutoPeftModelForCausalLM

# Reload the fine-tuned model
ft_model = AutoPeftModelForCausalLM.from_pretrained(
    OUTPUT_DIR,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
ft_tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)

# Test with a tricky chunk (title page / sparse content)
test_chunk = """# Chapter 5: Continuous Random Variables

A continuous random variable is a random variable whose cumulative distribution function is continuous everywhere.
Unlike discrete random variables, continuous random variables can take any value in an interval.
The probability density function (PDF) f(x) satisfies P(a <= X <= b) = integral from a to b of f(x)dx."""

messages = [
    {"role": "system", "content": "You are an expert at analyzing educational content and extracting important knowledge that students should remember."},
    {"role": "user", "content": f"Analyze the following text chunk and extract important knowledge as Q&A pairs.\n\nText chunk:\n{test_chunk}"},
]

inputs = ft_tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to(ft_model.device)
outputs = ft_model.generate(inputs, max_new_tokens=1024, temperature=0.7, do_sample=True)
response = ft_tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
print(response)

In [ ]:
if REPORT_TO == "wandb":
    wandb.finish()